# Lab 10 | Web Scraping – Books to Scrape

**Objective:** Scrape book information from [Books to Scrape](https://books.toscrape.com/) using `requests`, BeautifulSoup and pandas.

The final function, `scrape_books(min_rating, max_price)`, visits all catalogue pages, filters books by rating and price, visits matching product pages, and returns a DataFrame with the seven required columns.

**Required run:** minimum rating `4.0` and maximum price `£20`.

## 1. Import the libraries

In [ ]:
# Install these libraries once if they are not already installed:
# %pip install requests beautifulsoup4 pandas

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Identify the scraper as a browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

CATALOGUE_URL = "https://books.toscrape.com/"

# Convert the text rating shown in the HTML into a number
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

## 2. Connect to the website

In [ ]:
response = requests.get(CATALOGUE_URL, headers=HEADERS, timeout=30)

if response.status_code == 200:
    print("Connection successful!")
else:
    print("Connection failed:", response.status_code)

response.raise_for_status()

## 3. Parse the HTML and inspect the catalogue

In [ ]:
soup = BeautifulSoup(response.content, "html.parser")

# Find every book card on the first catalogue page
book_cards = soup.select("article.product_pod")
print("Books on the first page:", len(book_cards))

In [ ]:
# Inspect the first book
first_book = book_cards[0]

title = first_book.select_one("h3 a")["title"]
price_text = first_book.select_one("p.price_color").get_text(strip=True)
rating_word = first_book.select_one("p.star-rating")["class"][1]

print("Title:", title)
print("Price:", price_text)
print("Rating:", RATING_MAP[rating_word])

## 4. Convert and filter the catalogue information

In [ ]:
price_number = float(price_text.replace("£", ""))
rating = RATING_MAP[rating_word]

print("Price as a number:", price_number)
print("Rating as a number:", rating)

In [ ]:
# Try the required filters on the first catalogue page
min_rating = 4.0
max_price = 20.0

filtered_books_first_page = []

for book in book_cards:
    book_title = book.select_one("h3 a")["title"]
    book_price_text = book.select_one("p.price_color").get_text(strip=True)
    book_price = float(book_price_text.replace("£", ""))
    book_rating_word = book.select_one("p.star-rating")["class"][1]
    book_rating = RATING_MAP[book_rating_word]

    if book_rating >= min_rating and book_price <= max_price:
        filtered_books_first_page.append({
            "Title": book_title,
            "Price (£)": book_price,
            "Rating": book_rating
        })

print("Matching books on the first page:", len(filtered_books_first_page))
pd.DataFrame(filtered_books_first_page).head()

## 5. Understand pagination

In [ ]:
# Find the link to the next catalogue page
next_button = soup.select_one("li.next a")

if next_button:
    next_page = next_button["href"]
    next_url = urljoin(CATALOGUE_URL, next_page)
    print("Next page URL:", next_url)
else:
    print("There is no next page.")

## 6. Inspect an individual book page

In [ ]:
# Get the URL of the first book on the current catalogue page
first_book_link = first_book.select_one("h3 a")["href"]
book_url = urljoin(CATALOGUE_URL, first_book_link)

# Request and parse the individual product page
detail_response = requests.get(book_url, headers=HEADERS, timeout=30)
detail_response.raise_for_status()
detail_soup = BeautifulSoup(detail_response.content, "html.parser")

print("Product page:", detail_soup.title.get_text(strip=True))

In [ ]:
# Product details are stored in a table
product_table = detail_soup.select_one("table.table-striped")
product_details = {}

for row in product_table.select("tr"):
    key = row.select_one("th").get_text(strip=True)
    value = row.select_one("td").get_text(strip=True)
    product_details[key] = value

print("UPC:", product_details.get("UPC"))
print("Availability:", product_details.get("Availability"))

In [ ]:
# Extract the genre from the breadcrumb navigation
genre_links = detail_soup.select("ul.breadcrumb li a")
genre = genre_links[2].get_text(strip=True) if len(genre_links) > 2 else None

# Some books do not have a description, so handle that case
description_header = detail_soup.select_one("#product_description")
description = (
    description_header.find_next("p").get_text(strip=True)
    if description_header else None
 )

print("Genre:", genre)
print("Description:", description)

## 7. Build the required reusable scraping function

The exploratory cells above demonstrate how the website is structured. The function below combines those steps, loops through every catalogue page, and returns the requested seven-column DataFrame.

In [ ]:
def scrape_books(min_rating, max_price):
    """Scrape books matching a minimum star rating and maximum price.

    Parameters
    ----------
    min_rating : int or float
        Minimum rating, from 1 to 5.
    max_price : int or float
        Maximum book price in pounds.

    Returns
    -------
    pandas.DataFrame
        UPC, Title, Price (£), Rating, Genre, Availability, Description.
    """
    if not 1 <= min_rating <= 5:
        raise ValueError("min_rating must be between 1 and 5")
    if max_price < 0:
        raise ValueError("max_price cannot be negative")

    books_data = []
    current_url = CATALOGUE_URL

    with requests.Session() as session:
        session.headers.update(HEADERS)

        # Continue until the catalogue has no next page
        while current_url:
            page_response = session.get(current_url, timeout=30)
            page_response.raise_for_status()
            page_soup = BeautifulSoup(page_response.content, "html.parser")

            book_cards = page_soup.select("article.product_pod")

            for book in book_cards:
                title = book.select_one("h3 a")["title"]
                price_text = book.select_one("p.price_color").get_text(strip=True)
                price = float(price_text.replace("£", ""))
                rating_word = book.select_one("p.star-rating")["class"][1]
                rating = RATING_MAP[rating_word]

                # Only visit product pages for books matching both filters
                if rating >= min_rating and price <= max_price:
                    relative_link = book.select_one("h3 a")["href"]
                    book_url = urljoin(current_url, relative_link)

                    detail_response = session.get(book_url, timeout=30)
                    detail_response.raise_for_status()
                    detail_soup = BeautifulSoup(
                        detail_response.content, "html.parser"
                    )

                    # Extract UPC and availability from the product table
                    product_table = detail_soup.select_one("table.table-striped")
                    product_details = {}

                    if product_table:
                        for row in product_table.select("tr"):
                            key = row.select_one("th").get_text(strip=True)
                            value = row.select_one("td").get_text(strip=True)
                            product_details[key] = value

                    # Extract genre
                    genre_links = detail_soup.select("ul.breadcrumb li a")
                    genre = (
                        genre_links[2].get_text(strip=True)
                        if len(genre_links) > 2 else None
                    )

                    # Extract description when available
                    description_header = detail_soup.select_one(
                        "#product_description"
                    )
                    description = (
                        description_header.find_next("p").get_text(strip=True)
                        if description_header else None
                    )

                    books_data.append({
                        "UPC": product_details.get("UPC"),
                        "Title": title,
                        "Price (£)": price,
                        "Rating": rating,
                        "Genre": genre,
                        "Availability": product_details.get("Availability"),
                        "Description": description
                    })

            # Move to the next catalogue page, if one exists
            next_button = page_soup.select_one("li.next a")
            current_url = (
                urljoin(current_url, next_button["href"])
                if next_button else None
            )

    columns = [
        "UPC", "Title", "Price (£)", "Rating", "Genre",
        "Availability", "Description"
    ]
    return pd.DataFrame(books_data, columns=columns)

## 8. Run the function with the lab's required values

In [ ]:
books_df = scrape_books(min_rating=4.0, max_price=20.0)

print("Total matching books:", len(books_df))
print("Dataset shape:", books_df.shape)
display(books_df.head())

## 9. Validate the results

In [ ]:
# Check the required columns
expected_columns = [
    "UPC", "Title", "Price (£)", "Rating", "Genre",
    "Availability", "Description"
]

print("All required columns present:", list(books_df.columns) == expected_columns)
print("All ratings meet the minimum:", (books_df["Rating"] >= 4.0).all())
print("All prices meet the maximum:", (books_df["Price (£)"] <= 20.0).all())

# Review missing values
display(books_df.isna().sum())

## 10. Optional: Explore selected genres

In [ ]:
# The lab's required function has two parameters (min_rating and max_price).
# Genre is extracted for every matching book. You can filter genres afterwards.

# Example: show the genres available in the scraped results
print(books_df["Genre"].value_counts())

# Example: filter to selected genres (change these to genres in your results)
selected_genres = ["Travel", "Mystery"]
selected_genre_books = books_df[books_df["Genre"].isin(selected_genres)]
display(selected_genre_books.head())

## 11. Save the final dataset

In [ ]:
books_df.to_csv("scraped_books.csv", index=False)
print("CSV file saved successfully: scraped_books.csv")

## Conclusion

The scraper uses Requests to retrieve catalogue and product pages, BeautifulSoup to parse the HTML, and pandas to return the results in a structured DataFrame. The required function was run with a minimum rating of 4 stars and a maximum price of £20. The final dataset contains UPC, title, price, rating, genre, availability and description.